In [1]:
from pathlib import Path
import torch
p = Path('/kaggle/input/notebooks/harshalpimpalshende/notebook7d3a5ae6c7/recovery_seed41/best.pt')
assert p.is_file(), str(p)
c = torch.load(p, map_location='cpu', weights_only=False)
print('Source checkpoint:', c['epoch'], c['validation_accuracy'], c['parameters'])
print('GPU:', torch.cuda.is_available())


Source checkpoint: 41 0.9264000058174133 4480474
GPU: True


In [2]:
import os
import math
import time
import json
import random
import copy
import shutil
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

SEED = 42
EPOCHS = 160
STOP_AFTER = 160
BATCH_SIZE = 256
device = torch.device('cuda')
assert torch.cuda.is_available()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_num_threads(4)

data_root = Path('/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl')
experiment_dir = Path('/kaggle/working') / ('wrn28_56_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
experiment_dir.mkdir(exist_ok=False)
source_path = Path('/kaggle/working/.virtual_documents/__notebook_source__.ipynb')
if source_path.exists():
    shutil.copy2(source_path, experiment_dir / 'notebook_before_training.ipynb')
train_frame = pd.read_csv(data_root / 'train_labels.csv')
test_frame = pd.read_csv(data_root / 'test.csv')
sample_frame = pd.read_csv(data_root / 'sample_submission.csv')
assert len(train_frame) == 50000
assert len(test_frame) == len(sample_frame) == 2000
assert list(sample_frame.columns) == ['id', 'label']
assert sample_frame.id.equals(test_frame.id)
assert train_frame.label.between(0, 9).all()
train_indices, valid_indices = train_test_split(
    np.arange(len(train_frame)), test_size=5000,
    random_state=SEED, stratify=train_frame.label
)
np.savez(experiment_dir / 'split.npz', train=train_indices, valid=valid_indices)

def read_training_image(filename):
    with Image.open(data_root / 'train_images' / filename) as image:
        return np.array(image.convert('RGB'))

with ThreadPoolExecutor(max_workers=16) as pool:
    train_images = np.stack(list(pool.map(read_training_image, train_frame.filename)))
train_mean = train_images[train_indices].mean((0, 1, 2)) / 255
train_std = train_images[train_indices].std((0, 1, 2)) / 255
normal_mean = torch.tensor(train_mean, dtype=torch.float32, device=device)[None, :, None, None]
normal_std = torch.tensor(train_std, dtype=torch.float32, device=device)[None, :, None, None]
all_images = torch.from_numpy(train_images).to(device).permute(0, 3, 1, 2).contiguous()
all_labels = torch.tensor(train_frame.label.values, dtype=torch.long, device=device)
train_ids = torch.tensor(train_indices, device=device)
valid_ids = torch.tensor(valid_indices, device=device)
del train_images

class WideResidualBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.projection = None if cin == cout and stride == 1 else nn.Conv2d(cin, cout, 1, stride, bias=False)

    def forward(self, x):
        h = F.relu(self.bn1(x))
        shortcut = x if self.projection is None else self.projection(h)
        h = self.conv1(h)
        h = self.conv2(F.dropout(F.relu(self.bn2(h)), p=0.1, training=self.training))
        return h + shortcut

class WideResNet28(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        layers = []
        cin = 16
        for stage, cout in enumerate((56, 112, 224)):
            for block in range(4):
                layers.append(WideResidualBlock(cin, cout, 2 if stage > 0 and block == 0 else 1))
                cin = cout
        self.blocks = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(cin)
        self.fc = nn.Linear(cin, 10)
        for layer in self.modules():
            if isinstance(layer, nn.Conv2d):
                nn.init.kaiming_normal_(layer.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(layer, nn.BatchNorm2d):
                nn.init.ones_(layer.weight)
                nn.init.zeros_(layer.bias)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        x = F.adaptive_avg_pool2d(F.relu(self.bn(x)), 1).flatten(1)
        return self.fc(x)

network = WideResNet28().to(device).to(memory_format=torch.channels_last)
parameter_count = sum(p.numel() for p in network.parameters() if p.requires_grad)
assert parameter_count <= 5_000_000, parameter_count
average_network = copy.deepcopy(network).eval()
average_network.requires_grad_(False)
config = dict(seed=SEED, epochs=EPOCHS, stop_after=STOP_AFTER, batch_size=BATCH_SIZE,
              channels=[56, 112, 224], blocks_per_stage=4,
              parameters=parameter_count, train_mean=train_mean.tolist(),
              train_std=train_std.tolist(), validation_size=5000,
              optimizer='SGD', learning_rate=0.05, weight_decay=0.0005,
              momentum=0.9, ema_decay=0.995,
              mixup_alpha=0.2, cutmix_alpha=1.0, clean_final_epochs=15,
              pretrained=False, external_training_data=False)
(experiment_dir / 'config.json').write_text(json.dumps(config, indent=2))
references = """Architecture: Zagoruyko and Komodakis, Wide Residual Networks (2016), https://arxiv.org/abs/1605.07146
Mixup: Zhang et al., mixup: Beyond Empirical Risk Minimization (2018), https://arxiv.org/abs/1710.09412
CutMix: Yun et al., CutMix (2019), https://arxiv.org/abs/1905.04899
Implementation uses PyTorch; no external weights or datasets are loaded.
"""
(experiment_dir / 'references.txt').write_text(references)
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('Trainable parameters:', parameter_count, flush=True)
print('Training / validation:', len(train_ids), len(valid_ids), flush=True)
print('Output:', experiment_dir, flush=True)

seed_folder = Path('/kaggle/input/notebooks/harshalpimpalshende/notebook7d3a5ae6c7/recovery_seed41')
seed_checkpoint = torch.load(seed_folder / 'best.pt', map_location='cpu', weights_only=False)
saved_split = np.load(seed_folder / 'split.npz')
assert np.array_equal(saved_split['train'], train_indices)
assert np.array_equal(saved_split['valid'], valid_indices)
network.load_state_dict(seed_checkpoint['model'])
average_network.load_state_dict(seed_checkpoint['model'])
config['initial_checkpoint_epoch'] = seed_checkpoint['epoch']
config['initial_validation_accuracy'] = seed_checkpoint['validation_accuracy']
config['recovery_mode'] = 'Model weights restored; new optimizer and cosine schedule'
(experiment_dir / 'config.json').write_text(json.dumps(config, indent=2))
shutil.copy2(seed_folder / 'best.pt', experiment_dir / 'best.pt')
shutil.copy2(seed_folder / 'validation_probabilities.npy', experiment_dir / 'validation_probabilities.npy')
print('Recovered trained-from-scratch weights from epoch', seed_checkpoint['epoch'], flush=True)

def normalize_images(images):
    return (images.float().div(255) - normal_mean) / normal_std

def training_augmentation(images):
    images = images.float().div(255)
    n = len(images)
    images = torch.where(torch.rand(n, 1, 1, 1, device=device) < 0.5, images.flip(3), images)
    padded = F.pad(images, (4, 4, 4, 4), mode='reflect')
    offsets = torch.randint(0, 9, (n, 2), device=device)
    positions = torch.arange(32, device=device)
    rows = (offsets[:, 0, None] + positions)[..., None]
    cols = (offsets[:, 1, None] + positions)[:, None, :]
    images = padded[torch.arange(n, device=device)[:, None, None, None],
                    torch.arange(3, device=device)[None, :, None, None],
                    rows[:, None], cols[:, None]]
    brightness = 1 + (torch.rand(n, 1, 1, 1, device=device) - 0.5) * 0.4
    contrast = 1 + (torch.rand(n, 1, 1, 1, device=device) - 0.5) * 0.4
    saturation = 1 + (torch.rand(n, 1, 1, 1, device=device) - 0.5) * 0.4
    images = images * brightness
    center = images.mean((1, 2, 3), keepdim=True)
    images = (images - center) * contrast + center
    gray = (images * images.new_tensor([0.299, 0.587, 0.114])[None, :, None, None]).sum(1, keepdim=True)
    images = ((images - gray) * saturation + gray).clamp(0, 1)
    return ((images - normal_mean) / normal_std).contiguous(memory_format=torch.channels_last)

def mix_training_batch(images, labels, epoch):
    if epoch >= EPOCHS - 15:
        return images, labels, labels, 1.0
    choice = np.random.rand()
    order = torch.randperm(len(images), device=device)
    if choice < 0.5:
        lam = np.random.beta(1.0, 1.0)
        size = int(32 * math.sqrt(1 - lam))
        cx, cy = np.random.randint(32, size=2)
        left, right = max(0, cx - size // 2), min(32, cx + size // 2)
        top, bottom = max(0, cy - size // 2), min(32, cy + size // 2)
        images[:, :, top:bottom, left:right] = images[order, :, top:bottom, left:right]
        lam = 1 - (right - left) * (bottom - top) / 1024
        return images, labels, labels[order], lam
    if choice < 0.7:
        lam = np.random.beta(0.2, 0.2)
        return images * lam + images[order] * (1 - lam), labels, labels[order], lam
    return images, labels, labels, 1.0

@torch.no_grad()
def validation_predictions(net):
    net.eval()
    results = []
    for ids in valid_ids.split(256):
        images = normalize_images(all_images[ids]).contiguous(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.float16):
            probabilities = (net(images).float().softmax(1) + net(images.flip(3)).float().softmax(1)) / 2
        results.append(probabilities)
    probabilities = torch.cat(results)
    accuracy = (probabilities.argmax(1) == all_labels[valid_ids]).float().mean().item()
    return accuracy, probabilities

optimizer = torch.optim.SGD(network.parameters(), lr=0.05, momentum=0.9,
                            nesterov=True, weight_decay=5e-4)
steps_per_epoch = math.ceil(len(train_ids) / BATCH_SIZE)
def lr_multiplier(step):
    epoch_fraction = step / steps_per_epoch
    if epoch_fraction < 5:
        return 0.1 + 0.9 * epoch_fraction / 5
    progress = (epoch_fraction - 5) / (EPOCHS - 5)
    return 0.001 + 0.999 * (1 + math.cos(math.pi * progress)) / 2

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
scaler = torch.amp.GradScaler('cuda')
loss_function = nn.CrossEntropyLoss(label_smoothing=0.0)
history = []
best_accuracy = seed_checkpoint['validation_accuracy']
training_started = time.time()
probe = training_augmentation(all_images[train_ids[:8]])
assert probe.shape == (8, 3, 32, 32)
assert network(probe).shape == (8, 10)
assert torch.isfinite(probe).all()
print('Data and model checks passed. Continuing from saved trained-from-scratch weights.', flush=True)

for epoch in range(STOP_AFTER):
    epoch_started = time.time()
    network.train()
    order = train_ids[torch.randperm(len(train_ids), device=device)]
    loss_total = 0.0
    for ids in order.split(BATCH_SIZE):
        images = training_augmentation(all_images[ids])
        images, labels_a, labels_b, lam = mix_training_batch(images, all_labels[ids], epoch)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = network(images)
            loss = lam * loss_function(logits, labels_a) + (1 - lam) * loss_function(logits, labels_b)
        if not torch.isfinite(loss):
            raise RuntimeError('Non-finite loss; previous checkpoints are preserved.')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        with torch.no_grad():
            torch._foreach_lerp_(list(average_network.parameters()), list(network.parameters()), 1 - 0.995)
            for average_buffer, current_buffer in zip(average_network.buffers(), network.buffers()):
                average_buffer.copy_(current_buffer)
        loss_total += loss.item() * len(ids)

    accuracy, valid_probabilities = validation_predictions(network)
    average_accuracy, average_probabilities = validation_predictions(average_network)
    selected_network = average_network if average_accuracy > accuracy else network
    selected_accuracy = max(accuracy, average_accuracy)
    record = dict(epoch=epoch + 1, loss=loss_total / len(train_ids),
                  validation=accuracy, ema_validation=average_accuracy,
                  lr=optimizer.param_groups[0]['lr'],
                  seconds=time.time() - epoch_started)
    history.append(record)
    pd.DataFrame(history).to_csv(experiment_dir / 'history.csv', index=False)

    if selected_accuracy > best_accuracy:
        best_accuracy = selected_accuracy
        best_checkpoint = dict(model={k: v.detach().cpu() for k, v in selected_network.state_dict().items()},
                               epoch=epoch + 1, validation_accuracy=best_accuracy,
                               parameters=parameter_count, config=config,
                               selected='ema' if selected_network is average_network else 'raw')
        temporary_path = experiment_dir / 'best.tmp'
        torch.save(best_checkpoint, temporary_path)
        temporary_path.replace(experiment_dir / 'best.pt')
        np.save(experiment_dir / 'validation_probabilities.npy',
                (average_probabilities if selected_network is average_network else valid_probabilities).cpu().numpy())

    if (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
        torch.save(dict(model=network.state_dict(), ema=average_network.state_dict(),
                        optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                        scaler=scaler.state_dict(), epoch=epoch + 1,
                        best_accuracy=best_accuracy, history=history, config=config,
                        torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(),
                        numpy_rng=np.random.get_state(), python_rng=random.getstate()),
                   experiment_dir / 'last.pt')
    print(f"Epoch {epoch + 1:3d}/{EPOCHS}: loss {record['loss']:.4f}, "
          f"val {accuracy:.4f}, EMA {average_accuracy:.4f}, "
          f"best {best_accuracy:.4f}, {record['seconds']:.1f}s", flush=True)

best_checkpoint = torch.load(experiment_dir / 'best.pt', map_location=device, weights_only=False)
network.load_state_dict(best_checkpoint['model'])
verified_accuracy, _ = validation_predictions(network)
assert abs(verified_accuracy - best_checkpoint['validation_accuracy']) < 1e-6
network.eval()
def read_test_image(filename):
    with Image.open(data_root / 'test_images' / filename) as image:
        return np.array(image.convert('RGB'))
with ThreadPoolExecutor(max_workers=16) as pool:
    test_images = np.stack(list(pool.map(read_test_image, test_frame.filename)))
test_probabilities = []
with torch.no_grad():
    for start in range(0, len(test_images), 256):
        images = torch.from_numpy(test_images[start:start + 256]).to(device).permute(0, 3, 1, 2)
        images = normalize_images(images).contiguous(memory_format=torch.channels_last)
        with torch.autocast('cuda', dtype=torch.float16):
            probabilities = (network(images).float().softmax(1) + network(images.flip(3)).float().softmax(1)) / 2
        test_probabilities.append(probabilities.cpu())
test_probabilities = torch.cat(test_probabilities).numpy()
np.save(experiment_dir / 'test_probabilities.npy', test_probabilities)
submission = sample_frame.copy()
submission['label'] = test_probabilities.argmax(1).astype('int64')
assert list(submission.columns) == ['id', 'label']
assert submission.id.equals(sample_frame.id)
assert submission.id.is_unique and len(submission) == 2000
assert submission.label.between(0, 9).all()
assert not submission.isna().any().any()
submission.to_csv(experiment_dir / 'submission.csv', index=False)
print('Verified validation accuracy:', verified_accuracy, flush=True)
print('Final model parameters:', parameter_count, flush=True)
print('Submission saved:', experiment_dir / 'submission.csv', flush=True)
print('Total minutes:', (time.time() - training_started) / 60, flush=True)


shutil.copy2(experiment_dir / 'submission.csv', Path('/kaggle/working/submission.csv'))
result_summary = dict(validation_accuracy=verified_accuracy, parameters=parameter_count,
                      initial_checkpoint_epoch=seed_checkpoint['epoch'],
                      additional_epochs=EPOCHS, public_score=None,
                      checkpoint_epoch_in_recovery=best_checkpoint['epoch'])
(experiment_dir / 'summary.json').write_text(json.dumps(result_summary, indent=2))
print(json.dumps(result_summary, indent=2), flush=True)


GPU: Tesla T4
Trainable parameters: 4480474
Training / validation: 45000 5000
Output: /kaggle/working/wrn28_56_20261105_040139
Recovered trained-from-scratch weights from epoch 41
Data and model checks passed. Continuing from saved trained-from-scratch weights.
Epoch   1/160: loss 0.5690, val 0.9426, EMA 0.9304, best 0.9426, 21.3s
Epoch   2/160: loss 0.6377, val 0.9410, EMA 0.9302, best 0.9426, 18.0s
Epoch   3/160: loss 0.6043, val 0.9300, EMA 0.9354, best 0.9426, 18.5s
Epoch   4/160: loss 0.7341, val 0.9140, EMA 0.9372, best 0.9426, 19.0s
Epoch   5/160: loss 0.6565, val 0.8966, EMA 0.9354, best 0.9426, 20.0s
Epoch   6/160: loss 0.7381, val 0.9138, EMA 0.9350, best 0.9426, 21.1s
Epoch   7/160: loss 0.6943, val 0.8966, EMA 0.9290, best 0.9426, 20.2s
Epoch   8/160: loss 0.6827, val 0.8796, EMA 0.9258, best 0.9426, 19.5s
Epoch   9/160: loss 0.7237, val 0.9180, EMA 0.9150, best 0.9426, 20.1s
Epoch  10/160: loss 0.6840, val 0.9266, EMA 0.9320, best 0.9426, 20.8s
Epoch  11/160: loss 0.7540, 